<h3>Derma-CNN</h3>

In [3]:
import os

def check_train_data(image_dir, mask_dir, image_suffix=".jpg", mask_suffix="_segmentation.png"):
    unmatched_images = []
    matched = []

    # Список файлов
    image_files = [f for f in os.listdir(image_dir) if f.endswith(image_suffix)]
    mask_files = [f for f in os.listdir(mask_dir) if f.endswith(mask_suffix)]
    
    # Проверка совпадения
    for image_file in image_files:
        base_name = image_file.replace(image_suffix, "")
        mask_file = f"{base_name}{mask_suffix}"
        if mask_file in mask_files:
            matched.append((os.path.join(image_dir, image_file), os.path.join(mask_dir, mask_file)))
        else:
            unmatched_images.append(image_file)
    
    return matched, unmatched_images

# Пример использования
image_dir = "data/TRAINING_IMAGES"
mask_dir = "data/TRAINING_MASKS"

matched, unmatched_images = check_train_data(image_dir, mask_dir)

print(f"Совпавшие пары: {len(matched)}")
print(f"Несовпавшие изображения: {unmatched_images}")


Совпавшие пары: 2594
Несовпавшие изображения: []


In [5]:
import os
import numpy as np
from tensorflow.keras.utils import Sequence
from PIL import Image
from sklearn.model_selection import train_test_split

class DataGenerator(Sequence):
    def __init__(self, image_paths, mask_paths, batch_size=16, target_size=(224, 224), augmentations=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.target_size = target_size
        self.augmentations = augmentations

    def __len__(self):
        return len(self.image_paths) // self.batch_size

    def __getitem__(self, index):
        # Batch data
        batch_images = self.image_paths[index * self.batch_size:(index + 1) * self.batch_size]
        batch_masks = self.mask_paths[index * self.batch_size:(index + 1) * self.batch_size]

        images, masks = [], []

        for img_path, mask_path in zip(batch_images, batch_masks):
            # Load and preprocess images and masks using PIL
            image = Image.open(img_path)
            image = image.resize(self.target_size)
            image = np.array(image) / 255.0  # Normalize to [0, 1]

            mask = Image.open(mask_path).convert("L")  # Convert to grayscale
            mask = mask.resize(self.target_size)
            mask = np.array(mask)
            mask = (mask > 0).astype(np.float32)  # Binarize mask

            if self.augmentations:
                augmented = self.augmentations(image=image, mask=mask)
                image, mask = augmented['image'], augmented['mask']

            images.append(image)
            masks.append(mask)

        return np.array(images), np.array(masks)

# Получение путей к данным
train_image_dir = "data/TRAINING_IMAGES"
train_mask_dir = "data/TRAINING_MASKS"

train_images = sorted([os.path.join(train_image_dir, f) for f in os.listdir(train_image_dir)])
train_masks = sorted([os.path.join(train_mask_dir, f) for f in os.listdir(train_mask_dir)])

# Разделение данных
train_img_paths, val_img_paths, train_mask_paths, val_mask_paths = train_test_split(
    train_images, train_masks, test_size=0.2, random_state=42
)

# Создание генераторов
train_generator = DataGenerator(train_img_paths, train_mask_paths, batch_size=16, augmentations=None)
val_generator = DataGenerator(val_img_paths, val_mask_paths, batch_size=16)



In [6]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def unet_model(input_size=(224, 224, 3)):
    # Входной слой
    inputs = layers.Input(input_size)

    # Энкодер: последовательность свёрток и пулингов
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # Боттлнек
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)

    # Декодер: последовательность транспонированных свёрток и пропускных соединений
    u6 = layers.Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4], axis=-1)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)

    u7 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3], axis=-1)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)

    u8 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2], axis=-1)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)

    u9 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1], axis=-1)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)

    # Выходной слой
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    # Создание модели
    model = Model(inputs, outputs)
    return model

# Инициализация модели
model = unet_model()
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 56, 56,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 28, 28,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 28, 28,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 14, 14,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 14, 14,    │  4,719,616 │ max_pooling2d_3[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 14, 14,    │  9,438,208 │ conv2d_8[0][0]    │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 28, 28,    │  2,097,664 │ conv2d_9[0][0]    │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 28, 28,    │          0 │ conv2d_transpose

 Total params: 31,031,745 (118.38 MB)

 Trainable params: 31,031,745 (118.38 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
import tensorflow.keras.backend as K
#реализация Dice Loss
def dice_loss(y_true, y_pred):
    smooth = 1.0
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


In [8]:
#реализация метрики Dice Coefficient
def dice_coefficient(y_true, y_pred):
    smooth = 1.0
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


In [9]:
#настройка модели
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=dice_loss,
    metrics=[dice_coefficient]
)


In [10]:
#подготовка данных для обучения
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Генератор данных с аугментациями
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Создание генераторов для изображений и масок
train_image_gen = train_datagen.flow_from_directory(
    'data/TRAINING_IMAGES',
    target_size=(224, 224),
    batch_size=16,
    class_mode=None,
    seed=42
)

train_mask_gen = train_datagen.flow_from_directory(
    'data/TRAINING_MASKS',
    target_size=(224, 224),
    batch_size=16,
    class_mode=None,
    color_mode='grayscale',
    seed=42
)

val_image_gen = val_datagen.flow_from_directory(
    'data/VAL_IMAGES',
    target_size=(224, 224),
    batch_size=16,
    class_mode=None,
    seed=42
)

# Функция объединения изображений и масок
def combine_generator(image_gen, mask_gen):
    for img, mask in zip(image_gen, mask_gen):
        yield img, mask


Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


In [ ]:
# Исправленная часть с генераторами данных
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Используем ранее созданные списки путей (matched pairs)
train_img_paths, val_img_paths, train_mask_paths, val_mask_paths = train_test_split(
    [i[0] for i in matched], [i[1] for i in matched], test_size=0.2, random_state=42
)

# Функция для загрузки и предобработки изображений
def load_and_preprocess(img_path, mask_path, target_size=(224, 224)):
    # Загрузка изображения
    img = Image.open(img_path).resize(target_size)
    img = np.array(img) / 255.0
    
    # Загрузка маски
    mask = Image.open(mask_path).convert('L').resize(target_size)
    mask = (np.array(mask) > 0).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)  # Добавляем размерность канала
    
    return img, mask

# Генератор данных с аугментациями
class CustomDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, img_paths, mask_paths, batch_size=16, target_size=(224, 224), augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.target_size = target_size
        self.augment = augment
        self.indices = np.arange(len(self.img_paths))
        
        # Аугментации
        self.image_datagen = ImageDataGenerator(
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            shear_range=0.1,
            zoom_range=0.1,
            horizontal_flip=True,
            fill_mode='nearest'
        )
        
    def __len__(self):
        return len(self.img_paths) // self.batch_size
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_img = []
        batch_mask = []
        
        for i in batch_indices:
            img, mask = load_and_preprocess(self.img_paths[i], self.mask_paths[i], self.target_size)
            
            if self.augment:
                # Применяем одинаковые аугментации к изображению и маске
                stacked = np.concatenate([img, mask], axis=-1)
                augmented = self.image_datagen.random_transform(stacked)
                img = augmented[..., :3]
                mask = augmented[..., 3:]
                mask = (mask > 0.5).astype(np.float32)  # Бинаризуем после аугментации
                
            batch_img.append(img)
            batch_mask.append(mask)
            
        return np.array(batch_img), np.array(batch_mask)
    
    def on_epoch_end(self):
        np.random.shuffle(self.indices)

# Создание генераторов
train_gen = CustomDataGenerator(train_img_paths, train_mask_paths, augment=True)
val_gen = CustomDataGenerator(val_img_paths, val_mask_paths)

# Обучение модели
history = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    verbose=1
)

In [ ]:
#подготовка тестового генератора
# Генератор данных для тестов
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

test_image_gen = test_datagen.flow_from_directory(
    'data/test/images',
    target_size=(224, 224),
    batch_size=1,
    class_mode=None,
    shuffle=False,
    seed=42
)

test_mask_gen = test_datagen.flow_from_directory(
    'data/test/masks',
    target_size=(224, 224),
    batch_size=1,
    class_mode=None,
    color_mode='grayscale',
    shuffle=False,
    seed=42
)

test_gen = combine_generator(test_image_gen, test_mask_gen)


In [ ]:
#оценка модели
# Оценка на тестовом наборе
results = model.evaluate(test_gen, steps=len(test_image_gen))
print(f"Test Loss: {results[0]}")
print(f"Test Dice Coefficient: {results[1]}")


In [ ]:
#визуализируем
import matplotlib.pyplot as plt

def visualize_predictions(model, image_gen, mask_gen, num_samples=5):
    for i in range(num_samples):
        img = next(image_gen)[0]  # Получение изображения
        mask = next(mask_gen)[0]  # Истинная маска
        
        # Предсказание
        pred_mask = model.predict(img[np.newaxis, ...])[0]
        pred_mask = (pred_mask > 0.5).astype('float32')  # Бинаризация

        # Отображение
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.title("Original Image")
        plt.imshow(img)
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.title("True Mask")
        plt.imshow(mask.squeeze(), cmap='gray')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.title("Predicted Mask")
        plt.imshow(pred_mask.squeeze(), cmap='gray')
        plt.axis('off')

        plt.show()

# Визуализация
visualize_predictions(model, test_image_gen, test_mask_gen)
